In [1]:
review = (
    "My headphones stopped charging after two days. "
    "I would like a refund."
)

print(review)

My headphones stopped charging after two days. I would like a refund.


In [ ]:
import requests

DECISION_MODEL = "tev1:0.8b"
DECISION_URL = "http://localhost:11434/v1/systemone"

topic_question = {
    "type": "choice",
    "instructions": "Choose the main topic of this customer review.",
    "criteria": {
        "product_quality": "How well the product works or how durable it is.",
        "delivery": "Shipping speed, delivery, or package arrival.",
        "customer_service": "Interactions with support staff.",
        "other": "None of the listed topics is the main topic.",
    },
}

response = requests.post(
    DECISION_URL,
    json={
        "model": DECISION_MODEL,
        "state": review,
        "questions": {"topic": topic_question},
    },
    timeout=180,
)
response.raise_for_status()

result = response.json()
result

{'model': 'tev1:0.8b',
 'answers': {'topic': {'type': 'choice',
   'choice': 'customer_service',
   'probabilities': {'product_quality': 0.0071553666215673495,
    'delivery': 0.0001744423545244921,
    'customer_service': 0.832165231857151,
    'other': 0.1605049591667572},
   'confidence': 0.776220309142868}},
 'usage': {'input_tokens': 192, 'output_tokens': 0}}

In [6]:
import pandas as pd

answer = result["answers"]["topic"]

print("Selected topic:", answer["choice"])

pd.Series(
    answer["probabilities"],
    name="model_probability",
).sort_values(ascending=False)

Selected topic: customer_service


customer_service    0.832165
other               0.160505
product_quality     0.007155
delivery            0.000174
Name: model_probability, dtype: float64

In [7]:
df = pd.DataFrame({
    "review": [
        review,
        "The package arrived two weeks later than promised.",
        "Support kept transferring me without answering my question.",
        "The support agent explained everything clearly and solved my problem.",
        "I bought these as a gift and have not tried them yet.",
        "It arrived on time, but the device will not turn on.",
    ],
    "human_topic": [
        "product_quality",
        "delivery",
        "customer_service",
        "customer_service",
        "other",
        "product_quality",
    ],
})

df

,review,human_topic
0,My headphones stopped charging after two days....,product_quality
1,The package arrived two weeks later than promi...,delivery
2,Support kept transferring me without answering...,customer_service
3,The support agent explained everything clearly...,customer_service
4,I bought these as a gift and have not tried th...,other
5,"It arrived on time, but the device will not tu...",product_quality


In [8]:
predicted_topics = []
top_probabilities = []

for text in df["review"]:
    response = requests.post(
        DECISION_URL,
        json={
            "model": DECISION_MODEL,
            "state": text,
            "questions": {"topic": topic_question},
        },
        timeout=180,
    )
    response.raise_for_status()

    answer = response.json()["answers"]["topic"]

    predicted_topics.append(answer["choice"])
    top_probabilities.append(max(answer["probabilities"].values()))

df["model_topic"] = predicted_topics
df["top_probability"] = top_probabilities

df

,review,human_topic,model_topic,top_probability
0,My headphones stopped charging after two days....,product_quality,customer_service,0.832165
1,The package arrived two weeks later than promi...,delivery,delivery,0.998997
2,Support kept transferring me without answering...,customer_service,customer_service,0.926014
3,The support agent explained everything clearly...,customer_service,customer_service,0.992677
4,I bought these as a gift and have not tried th...,other,other,0.687672
5,"It arrived on time, but the device will not tu...",product_quality,delivery,0.524401


In [9]:
pd.crosstab(
    df["human_topic"],
    df["model_topic"],
    rownames=["Human label"],
    colnames=["Model label"],
)

Model label,customer_service,delivery,other
Human label,,,
customer_service,2,0,0
delivery,0,1,0
other,0,0,1
product_quality,1,1,0


In [10]:
disagreements = df["human_topic"] != df["model_topic"]

df.loc[
    disagreements,
    ["review", "human_topic", "model_topic", "top_probability"],
]

,review,human_topic,model_topic,top_probability
0,My headphones stopped charging after two days....,product_quality,customer_service,0.832165
5,"It arrived on time, but the device will not tu...",product_quality,delivery,0.524401


In [11]:
response = requests.post(
    DECISION_URL,
    json={
        "model": DECISION_MODEL,
        "state": review,
        "questions": {
            "topic": topic_question,
            "refund_requested": {
                "type": "noul",
                "instructions": (
                    "Does the customer explicitly request a refund?"
                ),
                "criteria": {
                    "true": "The customer explicitly asks for money back.",
                    "false": "The customer does not explicitly ask for money back.",
                },
            },
        },
    },
    timeout=180,
)
response.raise_for_status()

answers = response.json()["answers"]

print("Topic:", answers["topic"]["choice"])
print(
    "Estimated probability of a refund request:",
    answers["refund_requested"]["noul"],
)

Topic: customer_service
Estimated probability of a refund request: 0.6764491838204298


In [12]:
import ollama

LLM_MODEL = "gemma3:1b"

response = ollama.chat(
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": (
                "Summarize this customer review in one short sentence. "
                "Use only information stated in the review.\n\n"
                + review
            ),
        }
    ],
)

print(response["message"]["content"])

The customer is dissatisfied with their headphones, as they stopped charging after only two days and are requesting a refund.


In [13]:
import json

response = ollama.chat(
    model=LLM_MODEL,
    format="json",
    messages=[
        {
            "role": "user",
            "content": (
                "Extract information from the review below. "
                "Return a JSON object with exactly these keys: "
                "product, problem, requested_action. "
                "Use strings for stated information and null for missing information. "
                "Do not invent details.\n\n"
                + review
            ),
        }
    ],
)

print(response["message"]["content"])

{"product": "headphones", "problem": "stopped charging", "requested_action": "refund"}


In [14]:
extracted = json.loads(response["message"]["content"])

pd.DataFrame([extracted])

,product,problem,requested_action
0,headphones,stopped charging,refund
